# 04 — Final Evaluation and Report

Final evaluation on the held-out test set. The selected checkpoints, the decision thresholds and the preregistered comparisons are fixed on validation before any test access; here they are applied once to the test split.

In [ ]:
from pathlib import Path
import gc
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))

import torch

from notebooks.utility.classifier_experiment import (
    configure_environment,
    experiment_configuration,
    load_existing_outputs,
    run_test,
)
from notebooks.utility.classifier_dataset_builder import test_rows
from notebooks.utility.classifier_analysis import (
    build_all_validation_ensembles,
    compare_validation,
    ensemble_metric_table,
    plot_ensemble_curves,
    plot_ensemble_overview,
)
from notebooks.utility.classifier_protocol import ARCHITECTURES, CONDITIONS, SEEDS
from notebooks.utility.final_evaluation import generate_publication_report

# Make one GPU visible, using the first job's policy.
base_configuration = experiment_configuration(ROOT, ARCHITECTURES[0], CONDITIONS[0], SEEDS[0])
base_configuration['root'] = str(ROOT)
configure_environment(base_configuration)

## 1. Held-out test set

In [ ]:
test_data = test_rows(ROOT)
{
    'test_images': len(test_data),
    'test_patients': len({row['patient_id'] for row in test_data}),
    'positive_images': sum(1 for row in test_data if row['label'] == 1),
}

## 2. Test-set inference over the 24 runs

Apply each validation-selected `checkpoint_best` to the test split and write `test_predictions.csv` and `test_metrics.json` next to every run.

In [ ]:
RUN_TEST_INFERENCE = True

if RUN_TEST_INFERENCE:
    for architecture in ARCHITECTURES:
        for condition in CONDITIONS:
            for seed in SEEDS:
                configuration = experiment_configuration(ROOT, architecture, condition, seed)
                configuration['root'] = str(ROOT)
                checkpoint = load_existing_outputs(ROOT, configuration)['checkpoint']
                if checkpoint is None:
                    raise RuntimeError(
                        f'No trained checkpoint for {architecture}/{condition}/seed_{seed}.'
                    )
                result = run_test(ROOT, configuration, checkpoint, test_data)
                print(architecture, condition, seed, 'test PR-AUC', round(result['metrics']['pr_auc'], 4))
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
else:
    print('Test inference disabled: set RUN_TEST_INFERENCE = True to write test predictions.')

## 3. Test ensembles

The eight three-seed ensembles (mean image probability, patient-level aggregation), identical in construction to the validation ensembles.

In [ ]:
test_ensembles = build_all_validation_ensembles(ROOT, split='test')
len(test_ensembles)

## 4. Patient-level metrics

In [ ]:
test_comparison = compare_validation(ROOT, test_ensembles, split='test')
ensemble_metric_table(test_ensembles)

## 5. Preregistered comparisons (Holm family of eight)

In [ ]:
test_comparison['comparisons'], test_comparison['holm_correction']

## 6. Plots

In [ ]:
overview_figure = plot_ensemble_overview(test_ensembles)
curve_figure = plot_ensemble_curves(ROOT, test_ensembles, split='test')
(overview_figure, curve_figure)

## 7. Publication report

In [ ]:
report_path = generate_publication_report(ROOT)
print(report_path)

## Interpretation and limitations

Primary inference uses patient-level bootstrap and Holm correction over the eight declared comparisons on the test set. Any additional analysis is exploratory. Checkpoints and thresholds are fixed on validation, so the test set contributes no model selection.